In [3]:
import language_tool_python
from typing import List, Dict, Any

def check_grammar(text: str) -> List[Dict[str, Any]]:
    """
    Verifica a gramática de um texto em inglês e retorna os erros encontrados.

    Args:
        text: O texto a ser verificado (ex: uma entrada do diário).

    Returns:
        Uma lista de dicionários, onde cada dicionário representa um erro encontrado,
        contendo detalhes como a mensagem, sugestões e a posição do erro.
    """
    # Inicializa a ferramenta para o inglês americano. A primeira vez pode demorar
    # um pouco, pois ele baixa o servidor do LanguageTool.
    # 'en-US' para inglês americano, 'en-GB' para britânico.
    tool = language_tool_python.LanguageTool('en-US')
    
    matches = tool.check(text)
    
    # Vamos formatar a saída para ser mais útil para o nosso app
    found_errors = []
    for match in matches:
        error_info = {
            "rule": match.ruleId,
            "message": match.message,
            "suggestions": match.replacements,
            "context": match.context,
            "offset": match.offset,
            "length": match.errorLength
        }
        found_errors.append(error_info)
        
    return found_errors

if __name__ == "__main__":
    # Simula uma entrada de diário com vários tipos de erros
    diary_entry = """
Today were a good day. I feeled happy because I startd a new project.
My friend, she say that the idea is great. I need to buy alot of things for it.
This are the reasons why I am excited. There is many challenges ahead.
    """

    print("--- Verificando a Entrada do Diário ---")
    print(f"Texto Original:\n{diary_entry}")
    
    errors = check_grammar(diary_entry)
    
    print("\n--- Erros Gramaticais e de Estilo Encontrados ---")
    if not errors:
        print("Nenhum erro encontrado. Ótimo trabalho!")
    else:
        for error in errors:
            print(f"-> Mensagem: {error['message']}")
            print(f"   Contexto: ...{error['context']}...")
            print(f"   Sugestões: {error['suggestions']}")
            print(f"   Rule: {error['rule']}")
            print(f"   Offset: {error['offset']}")
            print(f"   Length: {error['length']}")

            print("-" * 20)

--- Verificando a Entrada do Diário ---
Texto Original:

Today were a good day. I feeled happy because I startd a new project.
My friend, she say that the idea is great. I need to buy alot of things for it.
This are the reasons why I am excited. There is many challenges ahead.
    

--- Erros Gramaticais e de Estilo Encontrados ---
-> Mensagem: Possible spelling mistake. Did you mean “felt”, the past tense form of the verb ‘feel’?
   Contexto: ... Today were a good day. I feeled happy because I startd a new project. M......
   Sugestões: ['felt', 'fueled', 'felled', 'heeled', 'keeled', 'peeled', 'reeled', 'feeler', 'fee led', 'Feeley']
   Rule: MORFOLOGIK_RULE_EN_US
   Offset: 26
   Length: 6
--------------------
-> Mensagem: Possible spelling mistake found.
   Contexto: ......re a good day. I feeled happy because I startd a new project. My friend, she say that ......
   Sugestões: ['started', 'start', 'starts', 'stared']
   Rule: MORFOLOGIK_RULE_EN_US
   Offset: 49
   Length: 6
------

In [4]:
import tkinter as tk
from tkinter import scrolledtext
import language_tool_python
from typing import List, Dict, Any

class IntelligentDiaryApp:
    """
    Uma aplicação de diário inteligente com verificação gramatical integrada.
    Versão 1.0: Destaque de erros e exibição de sugestões ao clicar.
    """
    def __init__(self, root: tk.Tk):
        self.root = root
        self.root.title("Diário Inteligente v1.0")
        self.root.geometry("800x600")

        # Configura a ferramenta de verificação.
        # A primeira inicialização pode ser lenta.
        print("Inicializando o verificador gramatical (pode levar um momento)...")
        self.tool = language_tool_python.LanguageTool('en-US')
        print("Verificador pronto.")

        # --- Layout da Interface ---
        self.main_frame = tk.Frame(root)
        self.main_frame.pack(pady=10, padx=10, fill="both", expand=True)

        # Widget de texto principal para a entrada do diário
        self.text_area = scrolledtext.ScrolledText(
            self.main_frame, 
            wrap=tk.WORD, 
            font=("Arial", 12),
            undo=True
        )
        self.text_area.pack(pady=5, padx=5, fill="both", expand=True)
        
        # Frame para os botões e a área de sugestões
        self.bottom_frame = tk.Frame(self.main_frame)
        self.bottom_frame.pack(fill="x", expand=False)

        # Botão para iniciar a verificação
        self.check_button = tk.Button(
            self.bottom_frame,
            text="Verificar Gramática",
            command=self.run_grammar_check
        )
        self.check_button.pack(side="left", padx=5, pady=5)
        
        # Label para mostrar as sugestões
        self.suggestion_label = tk.Label(
            self.bottom_frame,
            text="Sugestões aparecerão aqui...",
            fg="blue",
            font=("Arial", 10)
        )
        self.suggestion_label.pack(side="left", padx=10)

        # Armazena os erros encontrados para referência
        self.found_errors: List[Dict[str, Any]] = []

    def run_grammar_check(self):
        """
        Executa a verificação no texto atual, limpa destaques antigos
        e aplica os novos.
        """
        # Limpa formatações e eventos de cliques anteriores
        self.clear_highlights()
        
        text = self.text_area.get("1.0", tk.END)
        self.found_errors = self._check_grammar(text)
        
        if not self.found_errors:
            self.suggestion_label.config(text="Nenhum erro encontrado. Ótimo trabalho!")
            return
            
        self.suggestion_label.config(text=f"{len(self.found_errors)} erros encontrados. Clique em um erro para ver os detalhes.")
        
        # Aplica o destaque e o evento de clique para cada erro
        for i, error in enumerate(self.found_errors):
            tag_name = f"error_{i}"
            start_index = f"1.0 + {error['offset']} chars"
            end_index = f"1.0 + {error['offset'] + error['length']} chars"
            
            self.text_area.tag_add(tag_name, start_index, end_index)
            self.text_area.tag_config(tag_name, background="yellow", foreground="red")
            
            # Associa um evento de clique a esta tag específica
            self.text_area.tag_bind(tag_name, "<Button-1>", lambda event, e=error: self.show_suggestion(e))

    def _check_grammar(self, text: str) -> List[Dict[str, Any]]:
        """Função interna para chamar a ferramenta e formatar os resultados."""
        matches = self.tool.check(text)
        found_errors = []
        for match in matches:
            error_info = {
                "message": match.message,
                "suggestions": match.replacements,
                "offset": match.offset,
                "length": match.errorLength
            }
            found_errors.append(error_info)
        return found_errors

    def show_suggestion(self, error: Dict[str, Any]):
        """Atualiza o label de sugestões com a informação do erro clicado."""
        message = error['message']
        suggestions = ", ".join(error['suggestions'][:5]) # Mostra até 5 sugestões
        display_text = f"Erro: {message} | Sugestões: [{suggestions}]"
        self.suggestion_label.config(text=display_text)

    def clear_highlights(self):
        """Remove todas as tags de erro do widget de texto."""
        for tag in self.text_area.tag_names():
            if tag.startswith("error_"):
                self.text_area.tag_remove(tag, "1.0", tk.END)

if __name__ == "__main__":
    app_root = tk.Tk()
    app = IntelligentDiaryApp(app_root)
    app_root.mainloop()

Inicializando o verificador gramatical (pode levar um momento)...
Verificador pronto.


In [5]:
import tkinter as tk
from tkinter import scrolledtext, Menu
import language_tool_python
from typing import List, Dict, Any

class IntelligentDiaryApp:
    """
    Uma aplicação de diário inteligente com verificação gramatical integrada.
    Versão 2.0: Menu de contexto para correção rápida com um clique.
    """
    def __init__(self, root: tk.Tk):
        self.root = root
        self.root.title("Diário Inteligente v2.0")
        self.root.geometry("800x600")

        print("Inicializando o verificador gramatical (pode levar um momento)...")
        try:
            self.tool = language_tool_python.LanguageTool('en-US')
        except Exception as e:
            print(f"Erro ao inicializar o LanguageTool. Verifique a instalação do Java. Erro: {e}")
            self.tool = None
        print("Verificador pronto.")

        # --- Layout da Interface ---
        self.main_frame = tk.Frame(root)
        self.main_frame.pack(pady=10, padx=10, fill="both", expand=True)

        self.text_area = scrolledtext.ScrolledText(
            self.main_frame, wrap=tk.WORD, font=("Arial", 12), undo=True
        )
        self.text_area.pack(pady=5, padx=5, fill="both", expand=True)
        
        self.check_button = tk.Button(
            self.main_frame,
            text="Verificar Gramática",
            command=self.run_grammar_check
        )
        self.check_button.pack(pady=5)
        
        self.found_errors: List[Dict[str, Any]] = []

    def run_grammar_check(self):
        """Executa a verificação no texto e aplica os destaques e eventos."""
        if not self.tool:
            tk.messagebox.showerror("Erro", "Ferramenta de verificação não inicializada.")
            return

        self.clear_highlights()
        
        text = self.text_area.get("1.0", tk.END)
        self.found_errors = self._check_grammar(text)
        
        if not self.found_errors:
            tk.messagebox.showinfo("Resultado", "Nenhum erro encontrado. Ótimo trabalho!")
            return
            
        for i, error in enumerate(self.found_errors):
            tag_name = f"error_{i}"
            start_index = f"1.0 + {error['offset']} chars"
            end_index = f"1.0 + {error['offset'] + error['length']} chars"
            
            self.text_area.tag_add(tag_name, start_index, end_index)
            self.text_area.tag_config(tag_name, underline=True, foreground="red")
            
            # Associa o menu de correção ao clique direito do mouse
            self.text_area.tag_bind(tag_name, "<Button-3>", lambda event, e=error, t=tag_name: self.show_correction_menu(event, e, t))

    def show_correction_menu(self, event, error: Dict[str, Any], tag_name: str):
        """Cria e exibe um menu pop-up com as sugestões de correção."""
        menu = Menu(self.root, tearoff=0)
        
        # Adiciona o erro como um label no menu (desabilitado)
        menu.add_command(label=f"Erro: '{error['message']}'", state="disabled")
        menu.add_separator()

        # Adiciona cada sugestão como um item clicável no menu
        if error['suggestions']:
            for suggestion in error['suggestions'][:5]: # Limita a 5 sugestões
                menu.add_command(
                    label=suggestion,
                    command=lambda s=suggestion: self.apply_correction(s, error, tag_name)
                )
        else:
            menu.add_command(label="Nenhuma sugestão disponível", state="disabled")

        # Exibe o menu na posição do cursor do mouse
        try:
            menu.tk_popup(event.x_root, event.y_root)
        finally:
            menu.grab_release()

    def apply_correction(self, suggestion: str, error: Dict[str, Any], tag_name: str):
        """Substitui o texto errado pela sugestão escolhida."""
        start_index = f"1.0 + {error['offset']} chars"
        end_index = f"1.0 + {error['offset'] + error['length']} chars"
        
        self.text_area.delete(start_index, end_index)
        self.text_area.insert(start_index, suggestion)
        
        # Remove o destaque do erro que foi corrigido
        self.text_area.tag_remove(tag_name, "1.0", tk.END)

    def _check_grammar(self, text: str) -> List[Dict[str, Any]]:
        """Função interna para chamar a ferramenta."""
        matches = self.tool.check(text)
        return [{
            "message": match.message,
            "suggestions": match.replacements,
            "offset": match.offset,
            "length": match.errorLength
        } for match in matches]

    def clear_highlights(self):
        """Remove todas as tags de erro."""
        for tag in self.text_area.tag_names():
            if tag.startswith("error_"):
                self.text_area.tag_remove(tag, "1.0", tk.END)

if __name__ == "__main__":
    app_root = tk.Tk()
    app = IntelligentDiaryApp(app_root)
    app_root.mainloop()

Inicializando o verificador gramatical (pode levar um momento)...
Verificador pronto.


Exception in Tkinter callback
Traceback (most recent call last):
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\tkinter\__init__.py", line 1968, in __call__
    return self.func(*args)
           ^^^^^^^^^^^^^^^^
  File "C:\Users\guilh\AppData\Local\Temp\ipykernel_26268\4057376836.py", line 54, in run_grammar_check
    tk.messagebox.showinfo("Resultado", "Nenhum erro encontrado. Ótimo trabalho!")
    ^^^^^^^^^^^^^
AttributeError: module 'tkinter' has no attribute 'messagebox'
Exception in Tkinter callback
Traceback (most recent call last):
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\tkinter\__init__.py", line 1968, in __call__
    return self.func(*args)
           ^^^^^^^^^^^^^^^^
  File "C:\Users\guilh\AppData\Local\Temp\ipykernel_26268\4057376836.py", line 54, in run_grammar_check
    tk.messagebox.showinfo("Resultado", "Nenhum erro encontrado. Ótimo trabalh

In [1]:
import tkinter as tk
from tkinter import scrolledtext, Menu, messagebox
import language_tool_python
from typing import List, Dict, Any, Optional

class IntelligentDiaryApp:
    """
    Uma aplicação de diário inteligente com verificação gramatical integrada.
    Versão 3.0: Verificação automática em tempo real com "debouncing".
    """
    # Constante para o tempo de espera em milissegundos
    DEBOUNCE_DELAY_MS = 1500

    def __init__(self, root: tk.Tk):
        self.root = root
        self.root.title("Diário Inteligente v3.0 (Tempo Real)")
        self.root.geometry("800x600")

        self.tool = self._initialize_tool()

        # Variável para controlar o timer do debounce
        self.debounce_timer: Optional[str] = None
        self.found_errors: List[Dict[str, Any]] = []
        
        # --- Layout da Interface ---
        self.main_frame = tk.Frame(root, padx=10, pady=10)
        self.main_frame.pack(fill="both", expand=True)

        self.text_area = scrolledtext.ScrolledText(
            self.main_frame, wrap=tk.WORD, font=("Arial", 12), undo=True
        )
        self.text_area.pack(fill="both", expand=True)
        # Associa o evento de liberação de tecla à nossa função de debounce
        self.text_area.bind("<KeyRelease>", self.on_key_release)

        # Barra de status para feedback não intrusivo
        self.status_bar = tk.Label(root, text="Pronto.", bd=1, relief=tk.SUNKEN, anchor=tk.W)
        self.status_bar.pack(side=tk.BOTTOM, fill=tk.X)

    def _initialize_tool(self) -> Optional[language_tool_python.LanguageTool]:
        """Inicializa a ferramenta de linguagem de forma segura."""
        self.root.config(cursor="watch")
        self.root.update()
        print("Inicializando o verificador gramatical...")
        try:
            tool = language_tool_python.LanguageTool('en-US')
            print("Verificador pronto.")
            return tool
        except Exception as e:
            messagebox.showerror(
                "Erro de Inicialização",
                f"Não foi possível iniciar o LanguageTool. Verifique sua conexão e instalação do Java.\n\nErro: {e}"
            )
            return None
        finally:
            self.root.config(cursor="")

    def on_key_release(self, event=None):
        """Gerencia o timer de debounce a cada tecla liberada."""
        # Se um timer já estiver agendado, cancele-o
        if self.debounce_timer:
            self.root.after_cancel(self.debounce_timer)
        
        # Agenda a verificação para acontecer após o delay definido
        self.debounce_timer = self.root.after(self.DEBOUNCE_DELAY_MS, self.run_grammar_check)

    def run_grammar_check(self):
        """Executa a verificação, aplicando destaques e eventos."""
        if not self.tool:
            return

        self.status_bar.config(text="Verificando...")
        self.root.update_idletasks() # Força a atualização da UI antes da tarefa demorada
        
        self.clear_highlights()
        
        text = self.text_area.get("1.0", tk.END)
        self.found_errors = self._check_grammar(text)
        
        if not self.found_errors:
            self.status_bar.config(text="Nenhum erro encontrado.")
            return
            
        self.status_bar.config(text=f"{len(self.found_errors)} erros encontrados. Clique direito para corrigir.")
        
        for i, error in enumerate(self.found_errors):
            tag_name = f"error_{i}"
            start_index = f"1.0 + {error['offset']} chars"
            end_index = f"1.0 + {error['offset'] + error['length']} chars"
            
            self.text_area.tag_add(tag_name, start_index, end_index)
            self.text_area.tag_config(tag_name, underline=True, foreground="red")
            self.text_area.tag_bind(tag_name, "<Button-3>", lambda e, err=error, tag=tag_name: self.show_correction_menu(e, err, tag))

    def show_correction_menu(self, event, error: Dict[str, Any], tag_name: str):
        # (Esta função permanece idêntica à v2.0)
        menu = Menu(self.root, tearoff=0)
        menu.add_command(label=f"Erro: '{error['message']}'", state="disabled")
        menu.add_separator()
        if error['suggestions']:
            for suggestion in error['suggestions'][:5]:
                menu.add_command(label=suggestion, command=lambda s=suggestion: self.apply_correction(s, error, tag_name))
        else:
            menu.add_command(label="Nenhuma sugestão disponível", state="disabled")
        try:
            menu.tk_popup(event.x_root, event.y_root)
        finally:
            menu.grab_release()

    def apply_correction(self, suggestion: str, error: Dict[str, Any], tag_name: str):
        # (Esta função permanece idêntica à v2.0)
        start_index = f"1.0 + {error['offset']} chars"
        end_index = f"1.0 + {error['offset'] + error['length']} chars"
        self.text_area.delete(start_index, end_index)
        self.text_area.insert(start_index, suggestion)
        self.text_area.tag_remove(tag_name, "1.0", tk.END)

    def _check_grammar(self, text: str) -> List[Dict[str, Any]]:
        # (Esta função permanece idêntica à v2.0)
        matches = self.tool.check(text)
        return [{
            "message": match.message, "suggestions": match.replacements,
            "offset": match.offset, "length": match.errorLength
        } for match in matches]

    def clear_highlights(self):
        # (Esta função permanece idêntica à v2.0)
        for tag in self.text_area.tag_names():
            if tag.startswith("error_"):
                self.text_area.tag_remove(tag, "1.0", tk.END)

if __name__ == "__main__":
    app_root = tk.Tk()
    app = IntelligentDiaryApp(app_root)
    app_root.mainloop()

Inicializando o verificador gramatical...
Verificador pronto.


In [2]:
import tkinter as tk
from tkinter import scrolledtext, Menu, messagebox, filedialog
import language_tool_python
from typing import List, Dict, Any, Optional
from pathlib import Path

class IntelligentDiaryApp:
    """
    Uma aplicação de diário inteligente com verificação gramatical integrada.
    Versão 4.0: Funcionalidades de Salvar e Abrir arquivos.
    """
    DEBOUNCE_DELAY_MS = 1500

    def __init__(self, root: tk.Tk):
        self.root = root
        self.root.title("Diário Inteligente v4.0 - Novo Arquivo")
        self.root.geometry("800x600")

        self.tool = self._initialize_tool()
        self.debounce_timer: Optional[str] = None
        self.found_errors: List[Dict[str, Any]] = []
        self.current_file_path: Optional[Path] = None

        # --- Layout da Interface ---
        self._create_menubar()
        
        self.main_frame = tk.Frame(root, padx=10, pady=5)
        self.main_frame.pack(fill="both", expand=True)

        self.text_area = scrolledtext.ScrolledText(
            self.main_frame, wrap=tk.WORD, font=("Arial", 12), undo=True
        )
        self.text_area.pack(fill="both", expand=True)
        self.text_area.bind("<KeyRelease>", self.on_key_release)

        self.status_bar = tk.Label(root, text="Pronto.", bd=1, relief=tk.SUNKEN, anchor=tk.W)
        self.status_bar.pack(side=tk.BOTTOM, fill=tk.X)

    def _create_menubar(self):
        """Cria e configura a barra de menus da aplicação."""
        menubar = Menu(self.root)
        
        # --- Menu Arquivo ---
        file_menu = Menu(menubar, tearoff=0)
        file_menu.add_command(label="Novo", command=self.new_file, accelerator="Ctrl+N")
        file_menu.add_command(label="Abrir...", command=self.open_file, accelerator="Ctrl+O")
        file_menu.add_command(label="Salvar", command=self.save_file, accelerator="Ctrl+S")
        file_menu.add_command(label="Salvar como...", command=self.save_file_as)
        file_menu.add_separator()
        file_menu.add_command(label="Sair", command=self.root.quit)
        
        menubar.add_cascade(label="Arquivo", menu=file_menu)
        self.root.config(menu=menubar)
        
        # Adiciona atalhos de teclado (bônus)
        self.root.bind_all("<Control-n>", self.new_file)
        self.root.bind_all("<Control-o>", self.open_file)
        self.root.bind_all("<Control-s>", self.save_file)

    def new_file(self, event=None):
        """Limpa a área de texto para uma nova entrada."""
        self.text_area.delete("1.0", tk.END)
        self.current_file_path = None
        self.root.title("Diário Inteligente v4.0 - Novo Arquivo")
        self.status_bar.config(text="Pronto.")
        self.clear_highlights()

    def open_file(self, event=None):
        """Abre uma caixa de diálogo para carregar um arquivo de texto."""
        filepath = filedialog.askopenfilename(
            defaultextension=".txt",
            filetypes=[("Text Files", "*.txt"), ("All Files", "*.*")]
        )
        if not filepath:
            return
        
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                content = f.read()
            self.text_area.delete("1.0", tk.END)
            self.text_area.insert("1.0", content)
            self.current_file_path = Path(filepath)
            self.root.title(f"Diário Inteligente v4.0 - {self.current_file_path.name}")
            self.run_grammar_check() # Verifica o novo conteúdo
        except Exception as e:
            messagebox.showerror("Erro ao Abrir", f"Não foi possível ler o arquivo.\n\nErro: {e}")

    def save_file(self, event=None):
        """Salva o arquivo no caminho atual, ou pede um novo se não existir."""
        if self.current_file_path:
            try:
                content = self.text_area.get("1.0", tk.END)
                with open(self.current_file_path, 'w', encoding='utf-8') as f:
                    f.write(content)
                self.status_bar.config(text=f"Arquivo salvo em {self.current_file_path.name}")
            except Exception as e:
                messagebox.showerror("Erro ao Salvar", f"Não foi possível salvar o arquivo.\n\nErro: {e}")
        else:
            self.save_file_as()

    def save_file_as(self):
        """Abre uma caixa de diálogo para salvar o arquivo em um novo local."""
        filepath = filedialog.asksaveasfilename(
            defaultextension=".txt",
            filetypes=[("Text Files", "*.txt"), ("All Files", "*.*")]
        )
        if not filepath:
            return
        
        self.current_file_path = Path(filepath)
        self.save_file() # Chama a função save_file que agora tem o caminho
        self.root.title(f"Diário Inteligente v4.0 - {self.current_file_path.name}")

    # O resto do código (on_key_release, run_grammar_check, etc.) permanece idêntico à v3.0
    # ... (cole aqui o resto das funções da v3.0 para completar o script) ...
    def _initialize_tool(self) -> Optional[language_tool_python.LanguageTool]:
        """Inicializa a ferramenta de linguagem de forma segura."""
        self.root.config(cursor="watch")
        self.root.update()
        print("Inicializando o verificador gramatical...")
        try:
            tool = language_tool_python.LanguageTool('en-US')
            print("Verificador pronto.")
            return tool
        except Exception as e:
            messagebox.showerror(
                "Erro de Inicialização",
                f"Não foi possível iniciar o LanguageTool. Verifique sua conexão e instalação do Java.\n\nErro: {e}"
            )
            return None
        finally:
            self.root.config(cursor="")

    def on_key_release(self, event=None):
        if self.debounce_timer:
            self.root.after_cancel(self.debounce_timer)
        self.debounce_timer = self.root.after(self.DEBOUNCE_DELAY_MS, self.run_grammar_check)

    def run_grammar_check(self):
        if not self.tool: return
        self.status_bar.config(text="Verificando...")
        self.root.update_idletasks()
        self.clear_highlights()
        text = self.text_area.get("1.0", tk.END)
        self.found_errors = self._check_grammar(text)
        if not self.found_errors:
            self.status_bar.config(text="Nenhum erro encontrado.")
            return
        self.status_bar.config(text=f"{len(self.found_errors)} erros encontrados. Clique direito para corrigir.")
        for i, error in enumerate(self.found_errors):
            tag_name = f"error_{i}"
            start_index = f"1.0 + {error['offset']} chars"
            end_index = f"1.0 + {error['offset'] + error['length']} chars"
            self.text_area.tag_add(tag_name, start_index, end_index)
            self.text_area.tag_config(tag_name, underline=True, foreground="red")
            self.text_area.tag_bind(tag_name, "<Button-3>", lambda e, err=error, tag=tag_name: self.show_correction_menu(e, err, tag))

    def show_correction_menu(self, event, error: Dict[str, Any], tag_name: str):
        menu = Menu(self.root, tearoff=0)
        menu.add_command(label=f"Erro: '{error['message']}'", state="disabled")
        menu.add_separator()
        if error['suggestions']:
            for suggestion in error['suggestions'][:5]:
                menu.add_command(label=suggestion, command=lambda s=suggestion: self.apply_correction(s, error, tag_name))
        else:
            menu.add_command(label="Nenhuma sugestão disponível", state="disabled")
        try:
            menu.tk_popup(event.x_root, event.y_root)
        finally:
            menu.grab_release()

    def apply_correction(self, suggestion: str, error: Dict[str, Any], tag_name: str):
        start_index = f"1.0 + {error['offset']} chars"
        end_index = f"1.0 + {error['offset'] + error['length']} chars"
        self.text_area.delete(start_index, end_index)
        self.text_area.insert(start_index, suggestion)
        self.text_area.tag_remove(tag_name, "1.0", tk.END)

    def _check_grammar(self, text: str) -> List[Dict[str, Any]]:
        matches = self.tool.check(text)
        return [{"message": match.message, "suggestions": match.replacements, "offset": match.offset, "length": match.errorLength} for match in matches]

    def clear_highlights(self):
        for tag in self.text_area.tag_names():
            if tag.startswith("error_"):
                self.text_area.tag_remove(tag, "1.0", tk.END)


if __name__ == "__main__":
    app_root = tk.Tk()
    app = IntelligentDiaryApp(app_root)
    app_root.mainloop()

Inicializando o verificador gramatical...
Verificador pronto.


: 

In [1]:
import tkinter as tk
from tkinter import scrolledtext, Menu, messagebox, filedialog
import language_tool_python
from typing import List, Dict, Any, Optional
from pathlib import Path

class IntelligentDiaryApp:
    """
    Uma aplicação de diário inteligente com verificação gramatical integrada.
    Versão 5.0: Verificação de alterações não salvas.
    """
    DEBOUNCE_DELAY_MS = 1500

    def __init__(self, root: tk.Tk):
        self.root = root
        self.root.title("Diário Inteligente v5.0 - Novo Arquivo")
        self.root.geometry("850x650")

        self.tool = self._initialize_tool()
        self.debounce_timer: Optional[str] = None
        self.found_errors: List[Dict[str, Any]] = []
        self.current_file_path: Optional[Path] = None
        self.is_dirty = False # O "Dirty Flag"

        self._create_menubar()
        
        self.main_frame = tk.Frame(root, padx=10, pady=5)
        self.main_frame.pack(fill="both", expand=True)

        self.text_area = scrolledtext.ScrolledText(
            self.main_frame, wrap=tk.WORD, font=("Arial", 12), undo=True
        )
        self.text_area.pack(fill="both", expand=True)
        self.text_area.bind("<KeyRelease>", self.on_key_release)

        self.status_bar = tk.Label(root, text="Pronto.", bd=1, relief=tk.SUNKEN, anchor=tk.W)
        self.status_bar.pack(side=tk.BOTTOM, fill=tk.X)
        
        # Intercepta o clique no botão 'X' da janela
        self.root.protocol("WM_DELETE_WINDOW", self.on_closing)

    def _update_title(self):
        """Atualiza o título da janela com o nome do arquivo e o indicador '*'."""
        base_title = "Diário Inteligente v5.0"
        file_name = self.current_file_path.name if self.current_file_path else "Novo Arquivo"
        dirty_indicator = "*" if self.is_dirty else ""
        self.root.title(f"{base_title} - {dirty_indicator}{file_name}")
        
    def _set_dirty(self, dirty_state: bool):
        """Define o estado 'dirty' e atualiza o título da janela."""
        if self.is_dirty != dirty_state:
            self.is_dirty = dirty_state
            self._update_title()
            
    def _prompt_save_if_dirty(self) -> bool:
        """Verifica se há alterações. Se houver, pergunta ao usuário se deseja salvar.
        Retorna True se a ação puder continuar, False se for cancelada."""
        if not self.is_dirty:
            return True # Pode continuar, não há alterações

        file_name = self.current_file_path.name if self.current_file_path else "o arquivo atual"
        result = messagebox.askyesnocancel(
            "Salvar Alterações?",
            f"Deseja salvar as alterações em {file_name}?"
        )
        
        if result is True: # Sim (Salvar)
            return self.save_file()
        elif result is False: # Não (Não Salvar)
            return True
        else: # Cancelar
            return False

    def on_closing(self):
        """Lógica executada ao fechar a janela."""
        if self._prompt_save_if_dirty():
            self.root.destroy()
            
    def new_file(self, event=None):
        """Limpa a área de texto para uma nova entrada."""
        if not self._prompt_save_if_dirty():
            return

        self.text_area.delete("1.0", tk.END)
        self.current_file_path = None
        self._set_dirty(False)
        self.status_bar.config(text="Pronto.")
        self.clear_highlights()

    def open_file(self, event=None):
        """Abre uma caixa de diálogo para carregar um arquivo de texto."""
        if not self._prompt_save_if_dirty():
            return
            
        filepath = filedialog.askopenfilename(
            defaultextension=".txt", filetypes=[("Text Files", "*.txt"), ("All Files", "*.*")]
        )
        if not filepath:
            return
        
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                content = f.read()
            self.text_area.delete("1.0", tk.END)
            self.text_area.insert("1.0", content)
            self.current_file_path = Path(filepath)
            self._set_dirty(False)
            self.run_grammar_check()
        except Exception as e:
            messagebox.showerror("Erro ao Abrir", f"Não foi possível ler o arquivo.\n\nErro: {e}")

    def save_file(self, event=None) -> bool:
        """Salva o arquivo no caminho atual. Retorna True se bem-sucedido."""
        if self.current_file_path:
            try:
                content = self.text_area.get("1.0", tk.END)
                with open(self.current_file_path, 'w', encoding='utf-8') as f:
                    f.write(content)
                self.status_bar.config(text=f"Arquivo salvo em {self.current_file_path.name}")
                self._set_dirty(False)
                return True
            except Exception as e:
                messagebox.showerror("Erro ao Salvar", f"Não foi possível salvar o arquivo.\n\nErro: {e}")
                return False
        else:
            return self.save_file_as()

    def save_file_as(self) -> bool:
        """Abre uma caixa de diálogo para salvar. Retorna True se bem-sucedido."""
        filepath = filedialog.asksaveasfilename(
            defaultextension=".txt", filetypes=[("Text Files", "*.txt"), ("All Files", "*.*")]
        )
        if not filepath:
            return False # Usuário cancelou
        
        self.current_file_path = Path(filepath)
        return self.save_file()

    def on_key_release(self, event=None):
        """Gerencia o timer de debounce e marca o arquivo como 'dirty'."""
        self._set_dirty(True)
        if self.debounce_timer:
            self.root.after_cancel(self.debounce_timer)
        self.debounce_timer = self.root.after(self.DEBOUNCE_DELAY_MS, self.run_grammar_check)
        
    # O resto do código (run_grammar_check, etc.) permanece o mesmo da v4.0
    # ... (cole aqui as outras funções como _create_menubar, _initialize_tool, etc.) ...
    def _create_menubar(self):
        menubar = Menu(self.root)
        file_menu = Menu(menubar, tearoff=0)
        file_menu.add_command(label="Novo", command=self.new_file, accelerator="Ctrl+N")
        file_menu.add_command(label="Abrir...", command=self.open_file, accelerator="Ctrl+O")
        file_menu.add_command(label="Salvar", command=self.save_file, accelerator="Ctrl+S")
        file_menu.add_command(label="Salvar como...", command=self.save_file_as)
        file_menu.add_separator()
        file_menu.add_command(label="Sair", command=self.on_closing)
        menubar.add_cascade(label="Arquivo", menu=file_menu)
        self.root.config(menu=menubar)
        self.root.bind_all("<Control-n>", self.new_file)
        self.root.bind_all("<Control-o>", self.open_file)
        self.root.bind_all("<Control-s>", self.save_file)
        
    def _initialize_tool(self):
        self.root.config(cursor="watch")
        self.root.update()
        print("Inicializando o verificador gramatical...")
        try:
            tool = language_tool_python.LanguageTool('en-US')
            print("Verificador pronto.")
            return tool
        except Exception as e:
            messagebox.showerror("Erro de Inicialização", f"Não foi possível iniciar o LanguageTool. Verifique sua conexão e instalação do Java.\n\nErro: {e}")
            return None
        finally:
            self.root.config(cursor="")
            
    def run_grammar_check(self):
        if not self.tool: return
        self.status_bar.config(text="Verificando...")
        self.root.update_idletasks()
        self.clear_highlights()
        text = self.text_area.get("1.0", tk.END)
        self.found_errors = self._check_grammar(text)
        if not self.found_errors:
            self.status_bar.config(text="Nenhum erro encontrado.")
            return
        self.status_bar.config(text=f"{len(self.found_errors)} erros encontrados. Clique direito para corrigir.")
        for i, error in enumerate(self.found_errors):
            tag_name = f"error_{i}"
            start_index = f"1.0 + {error['offset']} chars"
            end_index = f"1.0 + {error['offset'] + error['length']} chars"
            self.text_area.tag_add(tag_name, start_index, end_index)
            self.text_area.tag_config(tag_name, underline=True, foreground="red")
            self.text_area.tag_bind(tag_name, "<Button-3>", lambda e, err=error, tag=tag_name: self.show_correction_menu(e, err, tag))

    def show_correction_menu(self, event, error, tag_name):
        menu = Menu(self.root, tearoff=0)
        menu.add_command(label=f"Erro: '{error['message']}'", state="disabled")
        menu.add_separator()
        if error['suggestions']:
            for suggestion in error['suggestions'][:5]:
                menu.add_command(label=suggestion, command=lambda s=suggestion: self.apply_correction(s, error, tag_name))
        else:
            menu.add_command(label="Nenhuma sugestão disponível", state="disabled")
        try:
            menu.tk_popup(event.x_root, event.y_root)
        finally:
            menu.grab_release()

    def apply_correction(self, suggestion, error, tag_name):
        start_index = f"1.0 + {error['offset']} chars"
        end_index = f"1.0 + {error['offset'] + error['length']} chars"
        self.text_area.delete(start_index, end_index)
        self.text_area.insert(start_index, suggestion)
        self.text_area.tag_remove(tag_name, "1.0", tk.END)

    def _check_grammar(self, text):
        matches = self.tool.check(text)
        return [{"message": match.message, "suggestions": match.replacements, "offset": match.offset, "length": match.errorLength} for match in matches]

    def clear_highlights(self):
        for tag in self.text_area.tag_names():
            if tag.startswith("error_"):
                self.text_area.tag_remove(tag, "1.0", tk.END)

if __name__ == "__main__":
    app_root = tk.Tk()
    app = IntelligentDiaryApp(app_root)
    app_root.mainloop()

Inicializando o verificador gramatical...
Verificador pronto.


: 

In [1]:
import tkinter as tk
from tkinter import scrolledtext, Menu, messagebox, filedialog
import language_tool_python
from typing import List, Dict, Any, Optional, Set
from pathlib import Path

class IntelligentDiaryApp:
    """
    Uma aplicação de diário inteligente com verificação gramatical integrada.
    Versão 6.0: Dicionário pessoal e lista de regras ignoradas.
    """
    DEBOUNCE_DELAY_MS = 1500
    DICTIONARY_FILE = Path("user_dict.txt")

    def __init__(self, root: tk.Tk):
        self.root = root
        self.root.title("Diário Inteligente v6.0 - Personalizável")
        self.root.geometry("850x650")

        self.tool = self._initialize_tool()
        
        # --- Novas variáveis de estado para personalização ---
        self.ignored_rules: Set[str] = set()
        self.user_dictionary: Set[str] = set()
        self._load_user_dictionary()

        self.debounce_timer: Optional[str] = None
        self.found_errors: List[Dict[str, Any]] = []
        self.current_file_path: Optional[Path] = None
        self.is_dirty = False

        self._create_menubar()
        
        self.main_frame = tk.Frame(root, padx=10, pady=5)
        self.main_frame.pack(fill="both", expand=True)

        self.text_area = scrolledtext.ScrolledText(
            self.main_frame, wrap=tk.WORD, font=("Arial", 12), undo=True
        )
        self.text_area.pack(fill="both", expand=True)
        self.text_area.bind("<KeyRelease>", self.on_key_release)

        self.status_bar = tk.Label(root, text="Pronto.", bd=1, relief=tk.SUNKEN, anchor=tk.W)
        self.status_bar.pack(side=tk.BOTTOM, fill=tk.X)
        
        self.root.protocol("WM_DELETE_WINDOW", self.on_closing)

    def _load_user_dictionary(self):
        """Carrega palavras do dicionário pessoal a partir de um arquivo."""
        if self.DICTIONARY_FILE.is_file():
            with open(self.DICTIONARY_FILE, 'r', encoding='utf-8') as f:
                self.user_dictionary = {line.strip().lower() for line in f if line.strip()}
            print(f"Dicionário pessoal carregado com {len(self.user_dictionary)} palavras.")

    def _add_word_to_dictionary_file(self, word: str):
        """Adiciona uma nova palavra ao arquivo do dicionário pessoal."""
        with open(self.DICTIONARY_FILE, 'a', encoding='utf-8') as f:
            f.write(f"{word}\n")

    def add_to_dictionary(self, word: str):
        """Adiciona uma palavra ao dicionário e atualiza a interface."""
        lower_word = word.lower()
        if lower_word not in self.user_dictionary:
            self.user_dictionary.add(lower_word)
            self._add_word_to_dictionary_file(lower_word)
            self.status_bar.config(text=f"'{word}' adicionada ao dicionário.")
            self.run_grammar_check() # Re-verifica o texto com a nova regra

    def ignore_rule(self, rule_id: str):
        """Adiciona uma regra à lista de ignorados e atualiza a interface."""
        self.ignored_rules.add(rule_id)
        self.status_bar.config(text=f"Regra '{rule_id}' ignorada nesta sessão.")
        self.run_grammar_check() # Re-verifica o texto com a nova regra

    def _check_grammar(self, text: str) -> List[Dict[str, Any]]:
        """Verifica a gramática e filtra os resultados com base nas regras do usuário."""
        raw_matches = self.tool.check(text)
        
        filtered_matches = []
        for match in raw_matches:
            # Filtra por regras ignoradas pelo usuário
            if match.ruleId in self.ignored_rules:
                continue
            
            # Filtra por palavras no dicionário pessoal (geralmente erros de ortografia)
            if 'MORFOLOGIK_RULE' in match.ruleId and match.matchedText.lower() in self.user_dictionary:
                continue

            # Se passar nos filtros, formata e adiciona à lista
            filtered_matches.append({
                "message": match.message,
                "suggestions": match.replacements,
                "offset": match.offset,
                "length": match.errorLength,
                "ruleId": match.ruleId,
                "matchedText": match.matchedText
            })
        return filtered_matches

    def show_correction_menu(self, event, error: Dict[str, Any], tag_name: str):
        """Cria e exibe um menu de contexto aprimorado."""
        menu = Menu(self.root, tearoff=0)
        
        # Sugestões de correção
        if error['suggestions']:
            for suggestion in error['suggestions'][:5]:
                menu.add_command(label=suggestion, command=lambda s=suggestion: self.apply_correction(s, error, tag_name))
        else:
            menu.add_command(label="Nenhuma sugestão disponível", state="disabled")
        
        menu.add_separator()
        
        # Opções de personalização
        error_word = error['matchedText']
        rule_id = error['ruleId']
        
        # Só mostra "adicionar ao dicionário" para erros de ortografia
        if 'MORFOLOGIK_RULE' in rule_id:
             menu.add_command(label=f"Adicionar '{error_word}' ao Dicionário", command=lambda w=error_word: self.add_to_dictionary(w))

        menu.add_command(label=f"Ignorar regra: {rule_id}", command=lambda r=rule_id: self.ignore_rule(r))

        menu.add_separator()
        menu.add_command(label=f"Info: {error['message']}", state="disabled")

        try:
            menu.tk_popup(event.x_root, event.y_root)
        finally:
            menu.grab_release()

    # O resto do código (new_file, open_file, etc.) permanece idêntico à v5.0
    # ... (cole aqui o resto das funções da v5.0 para completar o script) ...
    def _create_menubar(self):
        menubar = Menu(self.root)
        file_menu = Menu(menubar, tearoff=0)
        file_menu.add_command(label="Novo", command=self.new_file, accelerator="Ctrl+N")
        file_menu.add_command(label="Abrir...", command=self.open_file, accelerator="Ctrl+O")
        file_menu.add_command(label="Salvar", command=self.save_file, accelerator="Ctrl+S")
        file_menu.add_command(label="Salvar como...", command=self.save_file_as)
        file_menu.add_separator()
        file_menu.add_command(label="Sair", command=self.on_closing)
        menubar.add_cascade(label="Arquivo", menu=file_menu)
        self.root.config(menu=menubar)
        self.root.bind_all("<Control-n>", self.new_file)
        self.root.bind_all("<Control-o>", self.open_file)
        self.root.bind_all("<Control-s>", self.save_file)
        
    def _initialize_tool(self):
        self.root.config(cursor="watch")
        self.root.update()
        print("Inicializando o verificador gramatical...")
        try:
            tool = language_tool_python.LanguageTool('en-US')
            print("Verificador pronto.")
            return tool
        except Exception as e:
            messagebox.showerror("Erro de Inicialização", f"Não foi possível iniciar o LanguageTool. Verifique sua conexão e instalação do Java.\n\nErro: {e}")
            return None
        finally:
            self.root.config(cursor="")
            
    def _update_title(self):
        base_title = "Diário Inteligente v6.0"
        file_name = self.current_file_path.name if self.current_file_path else "Novo Arquivo"
        dirty_indicator = "*" if self.is_dirty else ""
        self.root.title(f"{base_title} - {dirty_indicator}{file_name}")
        
    def _set_dirty(self, dirty_state: bool):
        if self.is_dirty != dirty_state:
            self.is_dirty = dirty_state
            self._update_title()
            
    def _prompt_save_if_dirty(self) -> bool:
        if not self.is_dirty:
            return True
        file_name = self.current_file_path.name if self.current_file_path else "o arquivo atual"
        result = messagebox.askyesnocancel("Salvar Alterações?", f"Deseja salvar as alterações em {file_name}?")
        if result is True:
            return self.save_file()
        elif result is False:
            return True
        else:
            return False

    def on_closing(self):
        if self._prompt_save_if_dirty():
            self.root.destroy()
            
    def new_file(self, event=None):
        if not self._prompt_save_if_dirty():
            return
        self.text_area.delete("1.0", tk.END)
        self.current_file_path = None
        self._set_dirty(False)
        self.status_bar.config(text="Pronto.")
        self.clear_highlights()

    def open_file(self, event=None):
        if not self._prompt_save_if_dirty():
            return
        filepath = filedialog.askopenfilename(defaultextension=".txt", filetypes=[("Text Files", "*.txt"), ("All Files", "*.*")])
        if not filepath:
            return
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                content = f.read()
            self.text_area.delete("1.0", tk.END)
            self.text_area.insert("1.0", content)
            self.current_file_path = Path(filepath)
            self._set_dirty(False)
            self.run_grammar_check()
        except Exception as e:
            messagebox.showerror("Erro ao Abrir", f"Não foi possível ler o arquivo.\n\nErro: {e}")

    def save_file(self, event=None) -> bool:
        if self.current_file_path:
            try:
                content = self.text_area.get("1.0", tk.END)
                with open(self.current_file_path, 'w', encoding='utf-8') as f:
                    f.write(content)
                self.status_bar.config(text=f"Arquivo salvo em {self.current_file_path.name}")
                self._set_dirty(False)
                return True
            except Exception as e:
                messagebox.showerror("Erro ao Salvar", f"Não foi possível salvar o arquivo.\n\nErro: {e}")
                return False
        else:
            return self.save_file_as()

    def save_file_as(self) -> bool:
        filepath = filedialog.asksaveasfilename(defaultextension=".txt", filetypes=[("Text Files", "*.txt"), ("All Files", "*.*")])
        if not filepath:
            return False
        self.current_file_path = Path(filepath)
        return self.save_file()

    def on_key_release(self, event=None):
        self._set_dirty(True)
        if self.debounce_timer:
            self.root.after_cancel(self.debounce_timer)
        self.debounce_timer = self.root.after(self.DEBOUNCE_DELAY_MS, self.run_grammar_check)

    def run_grammar_check(self):
        if not self.tool: return
        self.status_bar.config(text="Verificando...")
        self.root.update_idletasks()
        self.clear_highlights()
        text = self.text_area.get("1.0", tk.END)
        self.found_errors = self._check_grammar(text)
        if not self.found_errors:
            self.status_bar.config(text="Nenhum erro encontrado.")
            return
        self.status_bar.config(text=f"{len(self.found_errors)} erros encontrados. Clique direito para corrigir.")
        for i, error in enumerate(self.found_errors):
            tag_name = f"error_{i}"
            start_index = f"1.0 + {error['offset']} chars"
            end_index = f"1.0 + {error['offset'] + error['length']} chars"
            self.text_area.tag_add(tag_name, start_index, end_index)
            self.text_area.tag_config(tag_name, underline=True, foreground="red")
            self.text_area.tag_bind(tag_name, "<Button-3>", lambda e, err=error, tag=tag_name: self.show_correction_menu(e, err, tag))

    def apply_correction(self, suggestion, error, tag_name):
        start_index = f"1.0 + {error['offset']} chars"
        end_index = f"1.0 + {error['offset'] + error['length']} chars"
        self.text_area.delete(start_index, end_index)
        self.text_area.insert(start_index, suggestion)
        self.text_area.tag_remove(tag_name, "1.0", tk.END)

    def clear_highlights(self):
        for tag in self.text_area.tag_names():
            if tag.startswith("error_"):
                self.text_area.tag_remove(tag, "1.0", tk.END)

if __name__ == "__main__":
    app_root = tk.Tk()
    app = IntelligentDiaryApp(app_root)
    app_root.mainloop()

Inicializando o verificador gramatical...
Verificador pronto.


In [2]:
import tkinter as tk
from tkinter import scrolledtext, Menu, messagebox, filedialog, Toplevel, Listbox, Scrollbar
from tkinter import ttk  # Import themed widgets
import language_tool_python
from typing import List, Dict, Any, Optional, Set
from pathlib import Path

# --- Nova Classe para a Janela de Configurações ---
class SettingsWindow(Toplevel):
    """
    Uma janela para gerenciar as configurações da aplicação, como o dicionário
    pessoal e as regras ignoradas.
    """
    def __init__(self, master, app_instance):
        super().__init__(master)
        self.transient(master)
        self.title("Configurações")
        self.geometry("500x400")
        
        self.app = app_instance # Referência à instância principal da aplicação

        # --- Criação das Abas ---
        notebook = ttk.Notebook(self)
        notebook.pack(pady=10, padx=10, fill="both", expand=True)

        dict_frame = ttk.Frame(notebook, padding="10")
        rules_frame = ttk.Frame(notebook, padding="10")

        notebook.add(dict_frame, text="Dicionário Pessoal")
        notebook.add(rules_frame, text="Regras Ignoradas")

        # --- Conteúdo da Aba "Dicionário Pessoal" ---
        self.dict_listbox = Listbox(dict_frame)
        dict_scrollbar = Scrollbar(dict_frame, orient="vertical", command=self.dict_listbox.yview)
        self.dict_listbox.config(yscrollcommand=dict_scrollbar.set)
        
        self._populate_dict_listbox()

        dict_scrollbar.pack(side="right", fill="y")
        self.dict_listbox.pack(side="left", fill="both", expand=True)
        
        remove_word_btn = ttk.Button(dict_frame, text="Remover Palavra Selecionada", command=self._remove_selected_word)
        remove_word_btn.pack(pady=5, side="bottom")

        # --- Conteúdo da Aba "Regras Ignoradas" ---
        self.rules_listbox = Listbox(rules_frame)
        rules_scrollbar = Scrollbar(rules_frame, orient="vertical", command=self.rules_listbox.yview)
        self.rules_listbox.config(yscrollcommand=rules_scrollbar.set)
        
        self._populate_rules_listbox()

        rules_scrollbar.pack(side="right", fill="y")
        self.rules_listbox.pack(side="left", fill="both", expand=True)
        
        reenable_rule_btn = ttk.Button(rules_frame, text="Reativar Regra Selecionada", command=self._reenable_selected_rule)
        reenable_rule_btn.pack(pady=5, side="bottom")

    def _populate_dict_listbox(self):
        self.dict_listbox.delete(0, tk.END)
        for word in sorted(list(self.app.user_dictionary)):
            self.dict_listbox.insert(tk.END, word)

    def _populate_rules_listbox(self):
        self.rules_listbox.delete(0, tk.END)
        for rule in sorted(list(self.app.ignored_rules)):
            self.rules_listbox.insert(tk.END, rule)

    def _remove_selected_word(self):
        selected_indices = self.dict_listbox.curselection()
        if not selected_indices:
            return
        selected_word = self.dict_listbox.get(selected_indices[0])
        self.app.remove_word_from_dictionary(selected_word)
        self._populate_dict_listbox() # Atualiza a lista na UI

    def _reenable_selected_rule(self):
        selected_indices = self.rules_listbox.curselection()
        if not selected_indices:
            return
        selected_rule = self.rules_listbox.get(selected_indices[0])
        self.app.reenable_rule(selected_rule)
        self._populate_rules_listbox() # Atualiza a lista na UI

# --- Classe Principal Modificada ---
class IntelligentDiaryApp:
    """
    Versão 7.0: Janela de configurações para gerenciar personalizações.
    """
    DEBOUNCE_DELAY_MS = 1500
    DICTIONARY_FILE = Path("user_dict.txt")

    def __init__(self, root: tk.Tk):
        # ... (O __init__ da v6.0 permanece quase idêntico) ...
        self.root = root
        self.root.title("Diário Inteligente v7.0 - Gerenciável")
        self.root.geometry("850x650")
        self.tool = self._initialize_tool()
        self.ignored_rules: Set[str] = set()
        self.user_dictionary: Set[str] = set()
        self._load_user_dictionary()
        self.debounce_timer: Optional[str] = None
        self.found_errors: List[Dict[str, Any]] = []
        self.current_file_path: Optional[Path] = None
        self.is_dirty = False
        self._create_menubar()
        self.main_frame = tk.Frame(root, padx=10, pady=5)
        self.main_frame.pack(fill="both", expand=True)
        self.text_area = scrolledtext.ScrolledText(self.main_frame, wrap=tk.WORD, font=("Arial", 12), undo=True)
        self.text_area.pack(fill="both", expand=True)
        self.text_area.bind("<KeyRelease>", self.on_key_release)
        self.status_bar = tk.Label(root, text="Pronto.", bd=1, relief=tk.SUNKEN, anchor=tk.W)
        self.status_bar.pack(side=tk.BOTTOM, fill=tk.X)
        self.root.protocol("WM_DELETE_WINDOW", self.on_closing)

    def _create_menubar(self):
        menubar = Menu(self.root)
        file_menu = Menu(menubar, tearoff=0)
        # ... (Comandos do menu de arquivo) ...
        file_menu.add_command(label="Novo", command=self.new_file, accelerator="Ctrl+N")
        file_menu.add_command(label="Abrir...", command=self.open_file, accelerator="Ctrl+O")
        file_menu.add_command(label="Salvar", command=self.save_file, accelerator="Ctrl+S")
        file_menu.add_command(label="Salvar como...", command=self.save_file_as)
        file_menu.add_separator()
        # --- Novo item de menu ---
        file_menu.add_command(label="Configurações...", command=self.open_settings_window)
        file_menu.add_separator()
        file_menu.add_command(label="Sair", command=self.on_closing)
        
        menubar.add_cascade(label="Arquivo", menu=file_menu)
        self.root.config(menu=menubar)
        # ... (atalhos) ...
        self.root.bind_all("<Control-n>", self.new_file)
        self.root.bind_all("<Control-o>", self.open_file)
        self.root.bind_all("<Control-s>", self.save_file)

    def open_settings_window(self):
        """Abre a janela de configurações."""
        SettingsWindow(self.root, self)

    def remove_word_from_dictionary(self, word: str):
        """Remove uma palavra do dicionário e reescreve o arquivo."""
        self.user_dictionary.discard(word.lower())
        self._rewrite_dictionary_file()
        self.run_grammar_check()

    def _rewrite_dictionary_file(self):
        """Reescreve o arquivo de dicionário com o conteúdo atual do set."""
        with open(self.DICTIONARY_FILE, 'w', encoding='utf-8') as f:
            for word in sorted(list(self.user_dictionary)):
                f.write(f"{word}\n")

    def reenable_rule(self, rule_id: str):
        """Remove uma regra da lista de ignorados."""
        self.ignored_rules.discard(rule_id)
        self.run_grammar_check()

    # O resto do código (new_file, open_file, etc.) permanece o mesmo da v6.0
    # ... (cole aqui as outras funções para completar o script) ...
    def _initialize_tool(self):
        self.root.config(cursor="watch")
        self.root.update()
        print("Inicializando o verificador gramatical...")
        try:
            tool = language_tool_python.LanguageTool('en-US')
            print("Verificador pronto.")
            return tool
        except Exception as e:
            messagebox.showerror("Erro de Inicialização", f"Não foi possível iniciar o LanguageTool. Verifique sua conexão e instalação do Java.\n\nErro: {e}")
            return None
        finally:
            self.root.config(cursor="")
            
    def _load_user_dictionary(self):
        if self.DICTIONARY_FILE.is_file():
            with open(self.DICTIONARY_FILE, 'r', encoding='utf-8') as f:
                self.user_dictionary = {line.strip().lower() for line in f if line.strip()}
            print(f"Dicionário pessoal carregado com {len(self.user_dictionary)} palavras.")

    def add_to_dictionary(self, word: str):
        lower_word = word.lower()
        if lower_word not in self.user_dictionary:
            self.user_dictionary.add(lower_word)
            with open(self.DICTIONARY_FILE, 'a', encoding='utf-8') as f:
                f.write(f"{lower_word}\n")
            self.status_bar.config(text=f"'{word}' adicionada ao dicionário.")
            self.run_grammar_check()

    def ignore_rule(self, rule_id: str):
        self.ignored_rules.add(rule_id)
        self.status_bar.config(text=f"Regra '{rule_id}' ignorada nesta sessão.")
        self.run_grammar_check()

    def _check_grammar(self, text: str) -> List[Dict[str, Any]]:
        raw_matches = self.tool.check(text)
        filtered_matches = []
        for match in raw_matches:
            if match.ruleId in self.ignored_rules:
                continue
            if 'MORFOLOGIK_RULE' in match.ruleId and match.matchedText.lower() in self.user_dictionary:
                continue
            filtered_matches.append({
                "message": match.message, "suggestions": match.replacements,
                "offset": match.offset, "length": match.errorLength,
                "ruleId": match.ruleId, "matchedText": match.matchedText
            })
        return filtered_matches

    def show_correction_menu(self, event, error, tag_name):
        menu = Menu(self.root, tearoff=0)
        if error['suggestions']:
            for suggestion in error['suggestions'][:5]:
                menu.add_command(label=suggestion, command=lambda s=suggestion: self.apply_correction(s, error, tag_name))
        else:
            menu.add_command(label="Nenhuma sugestão disponível", state="disabled")
        menu.add_separator()
        error_word = error['matchedText']
        rule_id = error['ruleId']
        if 'MORFOLOGIK_RULE' in rule_id:
             menu.add_command(label=f"Adicionar '{error_word}' ao Dicionário", command=lambda w=error_word: self.add_to_dictionary(w))
        menu.add_command(label=f"Ignorar regra: {rule_id}", command=lambda r=rule_id: self.ignore_rule(r))
        menu.add_separator()
        menu.add_command(label=f"Info: {error['message']}'", state="disabled")
        try:
            menu.tk_popup(event.x_root, event.y_root)
        finally:
            menu.grab_release()
            
    def _update_title(self):
        base_title = "Diário Inteligente v7.0"
        file_name = self.current_file_path.name if self.current_file_path else "Novo Arquivo"
        dirty_indicator = "*" if self.is_dirty else ""
        self.root.title(f"{base_title} - {dirty_indicator}{file_name}")
        
    def _set_dirty(self, dirty_state: bool):
        if self.is_dirty != dirty_state:
            self.is_dirty = dirty_state
            self._update_title()
            
    def _prompt_save_if_dirty(self) -> bool:
        if not self.is_dirty:
            return True
        file_name = self.current_file_path.name if self.current_file_path else "o arquivo atual"
        result = messagebox.askyesnocancel("Salvar Alterações?", f"Deseja salvar as alterações em {file_name}?")
        if result is True:
            return self.save_file()
        elif result is False:
            return True
        else:
            return False

    def on_closing(self):
        if self._prompt_save_if_dirty():
            self.root.destroy()
            
    def new_file(self, event=None):
        if not self._prompt_save_if_dirty():
            return
        self.text_area.delete("1.0", tk.END)
        self.current_file_path = None
        self._set_dirty(False)
        self.status_bar.config(text="Pronto.")
        self.clear_highlights()

    def open_file(self, event=None):
        if not self._prompt_save_if_dirty():
            return
        filepath = filedialog.askopenfilename(defaultextension=".txt", filetypes=[("Text Files", "*.txt"), ("All Files", "*.*")])
        if not filepath:
            return
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                content = f.read()
            self.text_area.delete("1.0", tk.END)
            self.text_area.insert("1.0", content)
            self.current_file_path = Path(filepath)
            self._set_dirty(False)
            self.run_grammar_check()
        except Exception as e:
            messagebox.showerror("Erro ao Abrir", f"Não foi possível ler o arquivo.\n\nErro: {e}")

    def save_file(self, event=None) -> bool:
        if self.current_file_path:
            try:
                content = self.text_area.get("1.0", tk.END)
                with open(self.current_file_path, 'w', encoding='utf-8') as f:
                    f.write(content)
                self.status_bar.config(text=f"Arquivo salvo em {self.current_file_path.name}")
                self._set_dirty(False)
                return True
            except Exception as e:
                messagebox.showerror("Erro ao Salvar", f"Não foi possível salvar o arquivo.\n\nErro: {e}")
                return False
        else:
            return self.save_file_as()

    def save_file_as(self) -> bool:
        filepath = filedialog.asksaveasfilename(defaultextension=".txt", filetypes=[("Text Files", "*.txt"), ("All Files", "*.*")])
        if not filepath:
            return False
        self.current_file_path = Path(filepath)
        return self.save_file()

    def on_key_release(self, event=None):
        self._set_dirty(True)
        if self.debounce_timer:
            self.root.after_cancel(self.debounce_timer)
        self.debounce_timer = self.root.after(self.DEBOUNCE_DELAY_MS, self.run_grammar_check)

    def run_grammar_check(self):
        # ... (código idêntico) ...
        if not self.tool: return
        self.status_bar.config(text="Verificando...")
        self.root.update_idletasks()
        self.clear_highlights()
        text = self.text_area.get("1.0", tk.END)
        self.found_errors = self._check_grammar(text)
        if not self.found_errors:
            self.status_bar.config(text="Nenhum erro encontrado.")
            return
        self.status_bar.config(text=f"{len(self.found_errors)} erros encontrados. Clique direito para corrigir.")
        for i, error in enumerate(self.found_errors):
            tag_name = f"error_{i}"
            start_index = f"1.0 + {error['offset']} chars"
            end_index = f"1.0 + {error['offset'] + error['length']} chars"
            self.text_area.tag_add(tag_name, start_index, end_index)
            self.text_area.tag_config(tag_name, underline=True, foreground="red")
            self.text_area.tag_bind(tag_name, "<Button-3>", lambda e, err=error, tag=tag_name: self.show_correction_menu(e, err, tag))

    def apply_correction(self, suggestion, error, tag_name):
        # ... (código idêntico) ...
        start_index = f"1.0 + {error['offset']} chars"
        end_index = f"1.0 + {error['offset'] + error['length']} chars"
        self.text_area.delete(start_index, end_index)
        self.text_area.insert(start_index, suggestion)
        self.text_area.tag_remove(tag_name, "1.0", tk.END)

    def clear_highlights(self):
        # ... (código idêntico) ...
        for tag in self.text_area.tag_names():
            if tag.startswith("error_"):
                self.text_area.tag_remove(tag, "1.0", tk.END)


if __name__ == "__main__":
    app_root = tk.Tk()
    app = IntelligentDiaryApp(app_root)
    app_root.mainloop()

Inicializando o verificador gramatical...
Verificador pronto.


: 